# PlaceMux Quality Baseline: Exploration Notebook

This notebook demonstrates the end-to-end pipeline: data loading → feature engineering → baseline scoring → model prediction → explainability → evaluation metrics.

In [1]:
import os, sys
import pandas as pd
import json
sys.path.append(os.path.abspath('../src'))
from data_loader import load_data
from feature_engineering import extract_features_for_pair
from baseline_matcher import calculate_baseline_score
from explainability import generate_explanation
import pickle

## 1. Load Data

In [2]:
candidates_df, jobs_df = load_data('../data')
print(f"Candidates: {len(candidates_df)}, Jobs: {len(jobs_df)}")
display(candidates_df.head())
display(jobs_df.head())

Candidates: 400, Jobs: 150


,candidate_id,name,skills,years_experience,education,certifications
0,10000,Hank 0,"Docker,C#,PowerBI,Data Analysis,AWS,Kubernetes",10.0,Master's,2
1,10001,Bob 1,"Kubernetes,Git,Data Analysis,C#,C++",9.0,Bachelor's,1
2,10002,Frank 2,"Kubernetes,Data Analysis,Git,C#,Java",6.0,Bachelor's,1
3,10003,Ivy 3,"Azure,C#,Python",2.0,Bachelor's,3
4,10004,Diana 4,"Data Analysis,SQL,PowerBI,Azure",8.0,Master's,0


,job_id,title,required_skills,minimum_experience,preferred_education
0,1000,Data Scientist,"React,PowerBI",2,Master's
1,1001,Data Scientist,"Python,SQL,Machine Learning,GCP,Node.js,Git",0,Bachelor's
2,1002,Data Analyst,"Node.js,Python,Kubernetes",2,Master's
3,1003,Machine Learning Engineer,"C++,Machine Learning,Java,AWS",1,Bachelor's
4,1004,Machine Learning Engineer,"Java,GCP,Node.js,PowerBI,AWS,Azure",4,Master's


## 2. Baseline Score Example

In [3]:
c = candidates_df.iloc[1].to_dict()
j = jobs_df.iloc[5].to_dict()

c_skills = [s.strip() for s in str(c.get('skills','')).split(',') if s.strip()]
j_skills = [s.strip() for s in str(j.get('required_skills','')).split(',') if s.strip()]

baseline = calculate_baseline_score(c_skills, j_skills)
print(f"Candidate: {c['name']} | Skills: {c_skills}")
print(f"Job: {j['title']} | Required: {j_skills}")
print(f"Baseline Score: {baseline}%")

Candidate: Bob 1 | Skills: ['Kubernetes', 'Git', 'Data Analysis', 'C#', 'C++']
Job: Machine Learning Engineer | Required: ['SQL', 'Java', 'C++', 'Machine Learning', 'Kubernetes', 'Data Analysis']
Baseline Score: 50.0%


## 3. Model Prediction + Explainability

In [4]:
with open('../models/baseline_model.pkl', 'rb') as f:
    artifact = pickle.load(f)
    model = artifact['model']
    scaler = artifact['scaler']

features = extract_features_for_pair(c, j)
X = pd.DataFrame([features])[['skill_overlap_percentage','experience_gap','education_match','certification_match_count','required_skill_coverage']]
X_scaled = scaler.transform(X)

pred = int(model.predict(X_scaled)[0])
prob = model.predict_proba(X_scaled)[0][1]
explanation = generate_explanation(features)

print(f"Prediction: {'Match' if pred else 'No Match'} (confidence: {prob*100:.1f}%)")
print("\nExplanation:")
for r in explanation:
    print(f"  • {r}")

Prediction: Match (confidence: 50.6%)

Explanation:
  • Partial skill match (50.0% of required skills met).
  • Experience requirement satisfied (exceeds by 8.0 years).
  • Did not meet preferred education level.


c:\Users\rushi\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rushi\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 4. Evaluation Metrics

These metrics were computed on the held-out 15% test split using `src/evaluate.py`. They represent an honest, non-circular evaluation of the baseline matching engine.

In [5]:
with open('../metrics/metrics.json', 'r') as f:
    metrics = json.load(f)

print("Evaluation Metrics (test set):")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

Evaluation Metrics (test set):
  precision: 0.8365
  recall: 0.6971
  accuracy: 0.9599
  f1_score: 0.7605
  false_positive_rate: 0.0137
